## PDF Generation Workflow

This notebook manages the end-to-end process of generating treated resumes. It combines job match data with randomized PII (Personally Identifiable Information) and specific design templates, then triggers an external n8n webhook to render the final PDFs.

## Workflow Overview

### 1. Configuration & Setup
*   **Environment Variables**: Loads sensitive credentials (API keys, Webhook URLs, MongoDB URI) from the `.env` file.
*   **Templates**: Defines the pool of available resume templates (ID + Markdown version) to be used.
*   **Treatments**: Sets the four experimental conditions: `control`, `Type_I`, `Type_II`, `Type_III`.

### 2. Data Loading
*   **Job Matches**: Reads the CSV file containing matched jobs (e.g., `cycle_14.1_matches.csv`).
*   **Country Mappings**: Loads `country_cluster_mapping.csv` to group countries into geographic clusters (e.g., "South Asia", "Sub-Saharan Africa").
*   **PII Records**: Loads `PII_country_cluster_mapping.csv` containing the pool of names and identities for each cluster.

### 3. Smart Filtering
*   **MongoDB Check**: Connects to the `n8n_resume_render_log` collection to find which job-treatment combinations have **already** been successfully rendered.
*   **Deduplication**: Filters the list of jobs to process, ensuring we don't re-generate PDFs that already exist (unless valid `REPROCESS_ALREADY_PROCESSED` is enabled).

### 4. PII & Template Assignment
*   **Clustering**: Determines the geographic cluster for each job's location.
*   **Randomization**:
    *   Selects 4 distinct PII records (one per treatment) from the cluster's pool.
    *   **Shuffles Templates**: Randomly assigns one of the 4 templates to each treatment condition to ensure design is not confounded with treatment.
    *   **Name Generation**: Consistently generates names (First + Last) and emails based on the selected PII record.

### 5. Execution (Webhook Trigger)
*   **Batch Processing**: Iterates through the prepared records in batches (default: 4 requests).
*   **Async Processing**: Sends a rich JSON payload to the **n8n Production Webhook**.
    *   Payload includes: Job details, assigned Name/Email/Phone, Markdown content, and selected Template ID.
*   **Rate Limiting**: Waits for a set interval (e.g., 130s) after every batch to respect API limits and allow n8n to complete rendering.
*   **Error Handling**: Monitors for consecutive failures and attempts to gracefully stop if the webhook is unresponsive.

### 6. Testing
*   Includes a **Single File Test** section to manually trigger the workflow for one specific file/treatment against a **Test Webhook** to verify logic before full production runs.

### Imports

In [ ]:
# Required imports
import requests
import json
import pandas as pd
import os
from datetime import datetime
import time  
import random
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
print("Test: ",os.getenv("TEST_WEBHOOK_URL"))
print("Prod: ",os.getenv("PRODUCTION_WEBHOOK_URL"))



### Configuration

In [ ]:
# Configuration Settings
# =====================

# Treatment Configuration
TREATMENT_TYPES = ['control', 'Type_I', 'Type_II', 'Type_III']

# Reprocessing Configuration
REPROCESS_ALREADY_PROCESSED = False  # Set to True to reprocess already processed files

# MongoDB Configuration
MONGODB_URI = os.getenv("MONGODB_URI")
DB_NAME = "Resume_study"
COLLECTION_NAME = "n8n_resume_render_log"

# API Configuration - Load from .env file only
TEST_WEBHOOK_URL = os.getenv('TEST_WEBHOOK_URL')
PRODUCTION_WEBHOOK_URL = os.getenv('PRODUCTION_WEBHOOK_URL')
WEBHOOK_USERNAME = os.getenv('WEBHOOK_USERNAME')
WEBHOOK_PASSWORD = os.getenv('WEBHOOK_PASSWORD')

# Validate that all required environment variables are set
required_env_vars = {
    'TEST_WEBHOOK_URL': TEST_WEBHOOK_URL,
    'PRODUCTION_WEBHOOK_URL': PRODUCTION_WEBHOOK_URL,
    'WEBHOOK_USERNAME': WEBHOOK_USERNAME,
    'WEBHOOK_PASSWORD': WEBHOOK_PASSWORD,
    'MONGODB_URI': MONGODB_URI
}

missing_vars = [var for var, value in required_env_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {missing_vars}")

AUTHORIZATION = (WEBHOOK_USERNAME, WEBHOOK_PASSWORD)

# Template Configuration
# Define the pool of 4 available templates to be shuffled
AVAILABLE_TEMPLATES = [
    {'template_id': '72b77b23d48f366e', 'markdown_template': 1},
    {'template_id': '73677b23d4896786', 'markdown_template': 2},
    {'template_id': '2ee77b23de5fc78e', 'markdown_template': 2},
    {'template_id': 'ca277b23d48328b0', 'markdown_template': 1}
]

print("Configuration Loaded:")
print(f"- Reprocess Already Processed: {REPROCESS_ALREADY_PROCESSED}")
print(f"- Treatment Types: {TREATMENT_TYPES}")
print(f"- MongoDB: {DB_NAME}.{COLLECTION_NAME}")
print(f"- Templates Configured: {len(AVAILABLE_TEMPLATES)}")


### Data Loading

In [ ]:
# Load Job Matches Data
# ====================

# Load the job matches data
file_path = "cycle_23_matches.csv"
job_matches_df = pd.read_csv(file_path)

# Rename columns to match our endpoint requirements
job_matches_df.rename(columns={'title': 'job_title'}, inplace=True)

print(f"Loaded {len(job_matches_df)} job matches")
print(f"Columns: {list(job_matches_df.columns)}")

# Display first few rows
print(f"\nSample data:")
job_matches_df.head()

### File Selection Logic

In [ ]:
# Updated File Selection Logic using MongoDB
# ==========================================

from pymongo import MongoClient

import dns.resolver
# Explicitly use Google's Public DNS to avoid local router DNS timeouts
dns.resolver.default_resolver = dns.resolver.Resolver(configure=False)
dns.resolver.default_resolver.nameservers = ['8.8.8.8', '8.8.4.4']
# -------------------------

print(f"Connecting to MongoDB...")

# Load existing results from MongoDB
processed_combinations = set()  # Store (job_id, treatment_type) strings or tuples

try:
    if not MONGODB_URI:
        raise ValueError("MONGODB_URI not set in configuration")
        
    client = MongoClient(MONGODB_URI)
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]
    
    # Query for successful renders
    # Filter for render_status='successful' and ensure required fields exist
    query = {
        "render_status": "successful",
        "job_posting_id": {"$exists": True},
        "treatment_type": {"$exists": True}
    }
    
    print(f"Querying {COLLECTION_NAME} for successful renders...")
    # Fetch job_posting_id and treatment_type directly
    cursor = collection.find(query, {"job_posting_id": 1, "treatment_type": 1})
    
    successful_renders = list(cursor)
    print(f"Found {len(successful_renders)} successful renders in MongoDB")
    
    if not REPROCESS_ALREADY_PROCESSED:
        for doc in successful_renders:
            job_id = doc.get('job_posting_id')
            treatment = doc.get('treatment_type')
            
            if job_id and treatment:
                processed_combinations.add((str(job_id), str(treatment)))
        
        print(f"Found {len(processed_combinations)} valid job-treatment combinations")

    client.close()

except Exception as e:
    print(f"Error connecting to or reading from MongoDB: {str(e)}")
    print("Will process all files as if none have been processed (unless this creates duplicates)")


# Filter job matches based on processing status
if not REPROCESS_ALREADY_PROCESSED and processed_combinations:
    print(f"\n Filtering job matches based on processing status...")
    
    # Create a list to store job matches that need processing
    jobs_to_process = []
    
    for _, job_row in job_matches_df.iterrows():
        # Match using job_posting_id or _id depending on what was used to construct u_id.
        job_id = str(job_row['job_posting_id']) # Changed from _id to job_posting_id based on user description
        
        processed_treatments = [t for j, t in processed_combinations if j == job_id]
        missing_treatments = [t for t in TREATMENT_TYPES if t not in processed_treatments]
        
        if missing_treatments:
            # This job needs processing for missing treatments
            jobs_to_process.append({
                'job_row': job_row,
                'missing_treatments': missing_treatments,
                'processed_treatments': processed_treatments
            })
    
    print(f" Jobs needing processing: {len(jobs_to_process)}")
    
    # Create filtered dataframe
    ids_to_process = [job['job_row']['_id'] for job in jobs_to_process]
    all_files_df = job_matches_df[job_matches_df['_id'].isin(ids_to_process)]
    
    print(f" Job matches to process: {len(all_files_df)}")
    
    total_missing_treatments = sum(len(job['missing_treatments']) for job in jobs_to_process)
    print(f" Total missing treatments to process: {total_missing_treatments}")
    
else:
    all_files_df = job_matches_df.copy()
    print(f" Processing all {len(all_files_df)} job matches")

# Display first 10 files with their details
print(f"\n Sample files to process (first 10):")
print("-" * 60)
for idx, row in all_files_df.head(10).iterrows():
    print(f"{idx+1:2d}. {row['file_id']}")
    print(f"    Job ID: {row.get('job_posting_id', 'N/A')}")
    if 'key_metrics.basics.likely_home_country' in row and pd.notna(row['key_metrics.basics.likely_home_country']):
        print(f"    Country: {row['key_metrics.basics.likely_home_country']}")
    print(f"    Job Title: {row.get('job_title', 'N/A')}")
    
    # Show treatment status
    if not REPROCESS_ALREADY_PROCESSED and processed_combinations:
        job_id = str(row['job_posting_id'])
        processed_treatments = [t for j, t in processed_combinations if j == job_id]
        missing_treatments = [t for t in TREATMENT_TYPES if t not in processed_treatments]
        print(f"    Processed: {processed_treatments}")
        print(f"    Missing: {missing_treatments}")
    print()

if len(all_files_df) > 10:
    print(f"... and {len(all_files_df) - 10} more files")

# Final Selection (Always All Files now)
files_to_process = all_files_df
print(f"\n Processing ALL {len(files_to_process)} job matches (filtered)")

print(f"\n Final selection: {len(files_to_process)} job matches to process")

# Calculate total operations
if not REPROCESS_ALREADY_PROCESSED and processed_combinations:
    total_operations = 0
    for _, job_row in files_to_process.iterrows():
        job_id = str(job_row['job_posting_id'])
        processed_treatments = [t for j, t in processed_combinations if j == job_id]
        missing_treatments = [t for t in TREATMENT_TYPES if t not in processed_treatments]
        total_operations += len(missing_treatments)
    print(f" Total operations to process (missing treatments only): {total_operations}")
else:
    total_operations = len(files_to_process) * len(TREATMENT_TYPES)
    print(f" Total operations to process: {total_operations}")


### PII Data Loading & Transformation

In [ ]:
# PII Data Loading and Functions
# =============================

# Load the country to geographic cluster mapping
country_cluster_df = pd.read_csv("country_cluster_mapping.csv")
print(f" Loaded country mapping: {len(country_cluster_df)} countries mapped to {country_cluster_df['Geographic_Cluster'].nunique()} clusters")

# Load the PII clusters data
pii_clusters_df = pd.read_csv("PII_country_cluster_mapping.csv")
print(f" Loaded PII clusters: {len(pii_clusters_df)} cluster-treatment combinations")

# Create lookup dictionaries for efficient access
country_to_cluster = dict(zip(country_cluster_df['Country'], country_cluster_df['Geographic_Cluster']))

# Create a nested dictionary for PII lookup: {cluster: [pii_records]}
pii_lookup = {}
for _, row in pii_clusters_df.iterrows():
    cluster = row['Geographic_Cluster']
    
    if cluster not in pii_lookup:
        pii_lookup[cluster] = []
    
    # Parse the name pools into lists
    male_names = [name.strip() for name in row['Male_First_Name_Pool'].split(',')]
    female_names = [name.strip() for name in row['Female_First_Name_Pool'].split(',')]
    
    pii_record = {
        'pii_id': row['PII_Identifier_ID'],
        'treatment_type': row['Treatment Type'],
        'last_name': row['Last_Name'],
        'email': row['Assigned_Email'],
        'phone': row['Assigned_Phone_Number'],
        'male_names': male_names,
        'female_names': female_names
    }
    
    pii_lookup[cluster].append(pii_record)

print(f"\n PII lookup structure created:")
for cluster in pii_lookup:
    print(f"  {cluster}: {len(pii_lookup[cluster])} PII records")

# Function to get cluster for a country with fallback to Anglosphere
def get_cluster_for_country(country):
    """
    Get geographic cluster for a country, with fallback to Anglosphere if not found.
    
    Args:
        country (str): Country name
    
    Returns:
        str: Geographic cluster name
    """
    if pd.isna(country) or country == 'Unknown' or country == '':
        print(f"  Country '{country}' is missing/unknown, using Anglosphere fallback")
        return 'Anglosphere'
    
    if country in country_to_cluster:
        return country_to_cluster[country]
    else:
        print(f"  Country '{country}' not found in mapping, using Anglosphere fallback")
        return 'Anglosphere'

# Helper function to generate PII from a selected record
def generate_pii_from_record(pii_record, used_first_names=None):
    """
    Generate PII data from a selected PII record.
    
    Args:
        pii_record (dict): Selected PII record
        used_first_names (set): Set of first names already used (to avoid duplicates)
    
    Returns:
        dict: Complete PII data with full_name, email, phone, etc.
    """
    if used_first_names is None:
        used_first_names = set()
    
    # Randomly select gender (50/50 chance)
    is_male = random.choice([True, False])
    
    # Get available names based on gender
    available_names = pii_record['male_names'] if is_male else pii_record['female_names']
    
    # Filter out already used names
    unused_names = [name for name in available_names if name not in used_first_names]
    
    # If all names are used, fall back to all available names (edge case)
    if not unused_names:
        print(f" Warning: All names used, allowing repeats")
        unused_names = available_names
    
    # Select random name from unused pool
    first_name = random.choice(unused_names)
    
    # Construct full name
    full_name = f"{first_name} {pii_record['last_name']}"
    
    return {
        'full_name': full_name,
        'email': pii_record['email'],
        'phone': pii_record['phone'],
        'first_name': first_name,
        'last_name': pii_record['last_name'],
        'gender': 'Male' if is_male else 'Female',
        'pii_id': pii_record['pii_id'],
        'pii_treatment_type': pii_record['treatment_type']
    }

# Test the functions with a sample
print(f"\n Testing PII generation:")
test_country = "India"
cluster = get_cluster_for_country(test_country)
print(f"  Country: {test_country} → Cluster: {cluster}")
print(f"  Available PII records: {len(pii_lookup[cluster])}")
for i, pii in enumerate(pii_lookup[cluster]):
    print(f"    {i+1}. {pii['pii_id']} ({pii['treatment_type']}) - {pii['last_name']}")

### Apply PII Data to Selected Files

In [ ]:
# Apply PII Data to Selected Files - All 4 PIIs per Job Match
# ==========================================================

# Apply PII selection to the files we want to process
print(" Generating PII data for selected files...")
print("-" * 60)

files_with_pii = []

for idx, row in files_to_process.iterrows():
    file_id = row['file_id']
    country = row.get('key_metrics.basics.likely_home_country', 'Unknown')
    
    print(f"\n File: {file_id}")
    print(f"   Country: {country}")
    
    # Get cluster for this country
    cluster = get_cluster_for_country(country)
    
    # Get all 4 PII records for this cluster
    cluster_pii_records = pii_lookup.get(cluster, [])
    
    if len(cluster_pii_records) != 4:
        print(f"     Warning: Expected 4 PII records for cluster '{cluster}', found {len(cluster_pii_records)}")
    
    if cluster_pii_records:
        # Shuffle the 4 PII records to randomize assignment
        shuffled_pii_records = cluster_pii_records.copy()
        random.shuffle(shuffled_pii_records)
        
        # NEW: Shuffle the templates for this specific file
        shuffled_templates = AVAILABLE_TEMPLATES.copy()
        random.shuffle(shuffled_templates)
        
        print(f"    Using all 4 PII records from cluster '{cluster}':")
        for i, pii_record in enumerate(shuffled_pii_records):
            print(f"     {i+1}. {pii_record['pii_id']} ({pii_record['treatment_type']}) - {pii_record['last_name']}")
        
        # Generate PII data for each treatment using the shuffled records
        treatment_pii_data = {}
        file_template_map = {}  # Store template assignments for this file
        used_first_names = set()  # Track used first names for this file
        
        for i, treatment_type in enumerate(TREATMENT_TYPES):
            # 1. Assign PII
            pii_record = shuffled_pii_records[i]
            pii_data = generate_pii_from_record(pii_record, used_first_names)  # ✅ PASS used names
            treatment_pii_data[treatment_type] = pii_data
            used_first_names.add(pii_data['first_name'])  # ✅ TRACK the name we just used
            
            # 2. Assign Template
            file_template_map[treatment_type] = shuffled_templates[i]
            
            print(f"   {treatment_type}: {treatment_pii_data[treatment_type]['full_name']} ({treatment_pii_data[treatment_type]['gender']})")
            print(f"     -> Template: {shuffled_templates[i]['template_id']} (MD: {shuffled_templates[i]['markdown_template']})")
        
        # Store the file data with PII AND Templates
        file_data = {
            'file_row': row,
            'pii_data': treatment_pii_data,
            'pii_ids': [pii['pii_id'] for pii in shuffled_pii_records],
            'template_assignments': file_template_map  
        }
        files_with_pii.append(file_data)
    else:
        print(f"    No PII records found for cluster '{cluster}'")
        continue

print(f"\n PII generation completed for {len(files_with_pii)} files")
print(f" Total operations to process: {len(files_with_pii)} files × {len(TREATMENT_TYPES)} treatments = {len(files_with_pii) * len(TREATMENT_TYPES)} operations")

### Audit Table: PII and Template Assignments

In [ ]:
# Audit Table: PII and Template Assignments
# =========================================

print(" AUDIT: PII and Template Assignments")
print("=" * 120)

# Create a list to store audit data
audit_data = []

for file_data in files_with_pii:
    file_row = file_data['file_row']
    match_id = file_row['_id']  #  Unique job match ID
    file_id = file_row['file_id']  # Resume filename (can repeat)
    country = file_row.get('key_metrics.basics.likely_home_country', 'Unknown')
    job_title = file_row.get('job_title', 'N/A')
    
    # Get all 4 treatments for this file
    for treatment_type in TREATMENT_TYPES:
        pii = file_data['pii_data'][treatment_type]
        template = file_data['template_assignments'][treatment_type]
        
        audit_data.append({
            'Match ID': match_id,  #  Unique job match
            'Resume File': file_id,  # Resume filename
            'Job Title': job_title,
            'Country': country,
            'Treatment': treatment_type,
            'First Name': pii['first_name'],
            'Last Name': pii['last_name'],
            'Full Name': pii['full_name'],
            'Gender': pii['gender'],
            'Email': pii['email'],
            'Phone': pii['phone'],
            'PII ID': pii['pii_id'],
            'Template ID': template['template_id'],
            'Markdown Template': template['markdown_template']
        })

# Convert to DataFrame for nice display
audit_df = pd.DataFrame(audit_data)

print(f"\n Audit table created with {len(audit_df)} rows ({len(files_with_pii)} job matches × 4 treatments)")
print("\n Sample (first 20 rows):")
print(audit_df.head(20).to_string(index=False))

#  Check for duplicate first names within same Match ID (_id)
print("\n🔍 Checking for duplicate first names within each job match (_id)...")
duplicate_check = audit_df.groupby('Match ID')['First Name'].apply(lambda x: len(x) != len(set(x)))
matches_with_duplicates = duplicate_check[duplicate_check].index.tolist()

if matches_with_duplicates:
    print(f"  WARNING: {len(matches_with_duplicates)} job matches have duplicate first names:")
    for match_id in matches_with_duplicates:
        match_names = audit_df[audit_df['Match ID'] == match_id]['First Name'].tolist()
        resume_file = audit_df[audit_df['Match ID'] == match_id]['Resume File'].iloc[0]
        print(f"   - Match {match_id} ({resume_file}): {match_names}")
else:
    print(" All job matches have unique first names across 4 treatments!")

# Optional: Show cases where same resume appears in multiple job matches
print("\n🔍 Checking for resumes matched to multiple jobs...")
resume_match_counts = audit_df.groupby('Resume File')['Match ID'].nunique()
multi_match_resumes = resume_match_counts[resume_match_counts > 1]

if len(multi_match_resumes) > 0:
    print(f" {len(multi_match_resumes)} resumes matched to multiple jobs:")
    for resume, count in multi_match_resumes.items():
        print(f"   - {resume}: {count} job matches")
else:
    print(" Each resume matched to exactly 1 job")

# Optional: Export to CSV for further analysis
# audit_df.to_csv('pii_template_audit.csv', index=False)
# print("\n Audit data exported to 'pii_template_audit.csv'")

# Display full table
print("\n Full Audit Table:")
audit_df

### Webhook Request Body Creation Function

In [ ]:
# Request Body Creation Function
# =============================

# Enhanced webhook request function that includes PII data
def create_enhanced_request_body(file_row, treatment_type, pii_data):
    """
    Create an enhanced request body with PII data.
    
    Args:
        file_row: Row from the job matches dataframe
        treatment_type: Treatment type being processed
        pii_data: PII data for this treatment type
    
    Returns:
        dict: Enhanced request body
    """
    # Handle location logic: Convert "Remote" to "Toronto, ON"
    location = file_row.get('location', 'Toronto, ON')
    if pd.isna(location) or location == 'Remote':
        location = 'Toronto, ON'
    
    request_body = {
        'id': file_row['_id'],
        'file_id': file_row['file_id'],
        'treatment_type': treatment_type,
        'name': pii_data['full_name'],
        'email': pii_data['email'],
        'phone': pii_data['phone'],
        'location': location,
        'first_name': pii_data['first_name'],
        'last_name': pii_data['last_name'],
        'gender': pii_data['gender'],
        'country': file_row.get('key_metrics.basics.likely_home_country', 'Unknown'),
        'geographic_cluster': country_to_cluster.get(file_row.get('key_metrics.basics.likely_home_country', ''), 'Unknown'),
        # Add job-related fields
        'company': file_row.get('company',''),
        'job_description': file_row.get('job_description', ''),
        'job_title': file_row.get('job_title', ''),
        'job_posting_id': file_row.get('job_posting_id', ''),
        'match_score': file_row.get('match_score', ''),
        'job_link': file_row.get('job_link', ''),
        'cycle': file_row.get('cycle', 0)
    }
    
    return request_body

# Test the request body creation
print(" Testing request body creation:")
print("-" * 60)

if files_with_pii:
    test_file = files_with_pii[0]
    test_treatment = TREATMENT_TYPES[0]
    test_pii = test_file['pii_data'][test_treatment]
    
    test_request = create_enhanced_request_body(
        test_file['file_row'], 
        test_treatment, 
        test_pii
    )
    
    print(f" Sample request body for {test_file['file_row']['file_id']} - {test_treatment}:")
    for key, value in test_request.items():
        if key == 'job_description':
            # Truncate job description for display
            display_value = value[:100] + "..." if len(str(value)) > 100 else value
            print(f"   {key}: {display_value}")
        else:
            print(f"   {key}: {value}")
else:
    print(" No files with PII data found. Please run the previous cell first.")

### Production - Concurrent Processing 

In [ ]:
# Production File Processing
# ======================================================

print("=== PRODUCTION PROCESSING ===")
print(f"Production webhook: {PRODUCTION_WEBHOOK_URL}")
print(f"Processing {len(files_with_pii)} files with {len(TREATMENT_TYPES)} treatments each")

# Initialize tracking variables
total_operations = len(files_with_pii) * len(TREATMENT_TYPES)
current_operation = 0
consecutive_errors = 0
max_consecutive_errors = 2  # Stop after 2 consecutive errors
success_count = 0
processing_stopped = False

# NEW: batch control – how many successful requests before we wait
BATCH_SIZE = 4
batch_success_count = 0

print(f"\n Starting processing")
print("=" * 60)

# Process files
for file_idx, file_data in enumerate(files_with_pii, 1):
    if processing_stopped:
        print(f"\n Processing already stopped due to consecutive errors")
        break
        
    # Extract file_row and pii_data from the correct structure
    file_row = file_data['file_row']
    file_id = file_row['file_id']
    pii_data = file_data['pii_data']
    
    print(f"\n Processing file {file_idx}/{len(files_with_pii)}: {file_id}")
    print("-" * 40)
    
    # Process treatments for this file
    for treatment_idx, treatment_type in enumerate(TREATMENT_TYPES, 1):
        if processing_stopped:
            print(f"   Stopping treatment processing due to consecutive errors")
            break
            
        current_operation += 1
        treatment_pii = pii_data[treatment_type]
        
        print(f"   Treatment {treatment_idx}/{len(TREATMENT_TYPES)}: {treatment_type}")
        print(f"   Using PII: {treatment_pii['full_name']} ({treatment_pii['email']})")
        
        # Get template configuration for this treatment type
        template_config = file_data['template_assignments'][treatment_type]
        print(f"      Template ID: {template_config['template_id']}")
        print(f"      Markdown Template: {template_config['markdown_template']}")
        
        # Create enhanced request body
        request_body = create_enhanced_request_body(file_row, treatment_type, treatment_pii)
        request_body['template_id'] = template_config['template_id']
        request_body['markdown_template'] = template_config['markdown_template']
        
        print(f" Sending request...")
        
        try:
            # Send request to n8n webhook (async mode - accepts immediate ack)
            response = requests.post(
                PRODUCTION_WEBHOOK_URL,
                json=request_body,
                auth=AUTHORIZATION,
                timeout=10  # Short timeout for immediate acceptance response
            )
            
            # Check for HTTP errors
            if response.status_code != 200:
                consecutive_errors += 1
                print(f"     HTTP Error: {response.status_code}")
                print(f"     Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
                
                if consecutive_errors >= max_consecutive_errors:
                    print(f"\nSTOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                    processing_stopped = True
                    break
                continue
            
            # Parse response JSON
            try:
                response_data = response.json()
            except json.JSONDecodeError:
                consecutive_errors += 1
                print(f"     JSON Parse Error")
                print(f"     Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
                
                if consecutive_errors >= max_consecutive_errors:
                    print(f"\nSTOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                    processing_stopped = True
                    break
                continue
            
            # Check for immediate acceptance (async processing mode)
            status = response_data.get('status', '').lower()
            
            if status == 'accepted':
                # Success - request accepted, processing will continue async
                consecutive_errors = 0
                success_count += 1
                batch_success_count += 1  # NEW: track successes in this batch
                
                print(f"     Request accepted (processing async)")
                print(f"     ID: {response_data.get('id', 'N/A')}")
                print(f"     Treatment: {response_data.get('treatment_type', 'N/A')}")
                print(f"     Message: {response_data.get('message', 'N/A')}")
                
                # Progress tracking
                progress_percent = (current_operation / total_operations) * 100
                print(f"     Progress: {current_operation}/{total_operations} ({progress_percent:.1f}%)")
                
                # NEW: Wait only after every BATCH_SIZE successful requests
                if current_operation < total_operations and not processing_stopped:
                    if batch_success_count % BATCH_SIZE == 0:
                        print(f"     Batch of {BATCH_SIZE} accepted. Waiting 130 seconds before next batch...")
                        time.sleep(130)  # Wait 130 seconds (2 min 10 sec) for processing to complete
                    else:
                        print(f"     ⏭ Skipping wait until batch of {BATCH_SIZE} is reached "
                              f"(current in batch: {batch_success_count % BATCH_SIZE})")
                
                continue  # Move to next request
            
            # Check for n8n workflow errors in response (fallback)
            if 'error' in status or 'failed' in status or 'exception' in status:
                consecutive_errors += 1
                print(f"     n8n Workflow Error: {status}")
                print(f"     Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
                
                if consecutive_errors >= max_consecutive_errors:
                    print(f"\n STOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                    processing_stopped = True
                    break
                continue
            
            # Fallback: Check for old-style response with PDF link (for backward compatibility)
            google_drive_link = response_data.get('Google_Drive_Link', '')
            file_id_response = response_data.get('id', '')
            file_name = response_data.get('file_name', '')
            timestamp = response_data.get('timestamp', '')
            
            if google_drive_link and file_id_response and file_name:
                # Old-style synchronous response (backward compatibility)
                consecutive_errors = 0
                success_count += 1
                batch_success_count += 1  # NEW: track successes in this batch
                
                print(f"     Success: PDF generated (synchronous mode)")
                print(f"        Google Drive Link: {google_drive_link}")
                print(f"        Filename: {file_name}")
                print(f"        ID: {file_id_response}")
                print(f"        Timestamp: {timestamp}")
                
                # Progress tracking
                progress_percent = (current_operation / total_operations) * 100
                print(f"     Progress: {current_operation}/{total_operations} ({progress_percent:.1f}%)")
                
                # NEW: Wait only after every BATCH_SIZE successful requests
                if current_operation < total_operations and not processing_stopped:
                    if batch_success_count % BATCH_SIZE == 0:
                        print(f"      Batch of {BATCH_SIZE} accepted. Waiting 130 seconds before next batch...")
                        time.sleep(130)  # Wait 130 seconds (2 min 10 sec) for processing to complete
                    else:
                        print(f"     ⏭ Skipping wait until batch of {BATCH_SIZE} is reached "
                              f"(current in batch: {batch_success_count % BATCH_SIZE})")
                
                continue
            
            # If we get here, response is neither "accepted" nor old-style success
            # Unknown response format
            consecutive_errors += 1
            print(f"     Unknown response format")
            print(f"     Response: {json.dumps(response_data, indent=2)[:200]}...")
            print(f"     Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
            
            if consecutive_errors >= max_consecutive_errors:
                print(f"\n STOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                processing_stopped = True
                break
            continue
            
        except requests.exceptions.Timeout:
            consecutive_errors += 1
            print(f"      Request Timeout (webhook didn't respond in 10s)")
            print(f"      Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
            
            if consecutive_errors >= max_consecutive_errors:
                print(f"\n STOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                processing_stopped = True
                break
            continue
            
        except Exception as e:
            consecutive_errors += 1
            print(f"      Exception: {str(e)}")
            print(f"      Consecutive errors: {consecutive_errors}/{max_consecutive_errors}")
            
            if consecutive_errors >= max_consecutive_errors:
                print(f"\n STOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
                processing_stopped = True
                break
            continue
    
    # Check if we should stop due to consecutive errors (after processing all treatments for a file)
    if processing_stopped:
        print(f"\n STOPPING EXECUTION: {consecutive_errors} consecutive errors detected!")
        print(f"   Processing stopped at file {file_idx}/{len(files_with_pii)}")
        break

# Final summary
print(f"\n Processing Summary:")
print(f"   Successful: {success_count}")
print(f"   Success Rate: {(success_count/current_operation*100):.1f}%" if current_operation > 0 else "N/A")

if processing_stopped:
    print(f"\n EXECUTION STOPPED EARLY DUE TO ERRORS!")
    print(f"   Consecutive errors: {consecutive_errors}")
    print(f"   Check n8n logs for detailed error information.")
    print(f"   Fix the issues and restart the script to continue processing.")
else:
    print(f"\n🎉 Processing workflow completed successfully!")

print(f" Total operations processed: {current_operation}")

### Single File Test (Test Webhook)

In [ ]:
# Single File Test
# ===============

print("=== SINGLE FILE TEST ===")
print(f"Test webhook: {TEST_WEBHOOK_URL}")

# Get first file for testing
if 'files_with_pii' in locals() and len(files_with_pii) > 0:
    # test_file = files_with_pii[1]
    test_treatment = 'Type_III' # 'control', 'Type_I', 'Type_II', 'Type_III'    
    # Get specific job ID to test
    test_id = "69a8eeac377a20a8545131e1"
    
    # Find the file with matching ID
    test_file = next((f for f in files_with_pii if f['file_row']['_id'] == test_id), files_with_pii[0])
    
    file_row = test_file['file_row']
    file_id = file_row['file_id']
    treatment_pii = test_file['pii_data'][test_treatment]
    
    print(f"\n Testing file: {file_id}")
    print(f" Treatment: {test_treatment}")
    print(f" PII: {treatment_pii['full_name']} ({treatment_pii['email']})")
    
    # Get template configuration
    # ✅ MANUALLY SELECT TEMPLATE (change index 0-3 to select different templates)
    template_config = AVAILABLE_TEMPLATES[1]  # 0=72b77b23d48f366e(MD:1), 1=73677b23d4896786(MD:2), 2=2ee77b23de5fc78e(MD:2), 3=ca277b23d48328b0(MD:1)
    print(f" Template ID: {template_config['template_id']}")
    print(f" Markdown Template: {template_config['markdown_template']}")
    
    # Create request body
    request_body = create_enhanced_request_body(file_row, test_treatment, treatment_pii)
    request_body['template_id'] = template_config['template_id']
    request_body['markdown_template'] = template_config['markdown_template']
    
    print(f"\n Sending test request...")
    
    try:
        # Send request
        response = requests.post(
            TEST_WEBHOOK_URL, #PRODUCTION_WEBHOOK_URL - CHANGE FROM TEST TO PRODUCTION WHEN RERUNNING SINGLE FILE AFTER A PRODUCTION CYCLE
            json=request_body,
            auth=AUTHORIZATION,
            timeout=120
        )
        
        print(f" Response Status: {response.status_code}")
        
        if response.status_code == 200:
            response_data = response.json()
            
            # Display response data
            print(f"\n Test successful!")
            print(f" Response Data:")
            print(f"   Status: {response_data.get('status', 'N/A')}")
            print(f"   ID: {response_data.get('id', 'N/A')}")
            print(f"   Filename: {response_data.get('file_name', 'N/A')}")
            print(f"   Timestamp: {response_data.get('timestamp', 'N/A')}")
            
            # Google Drive link
            google_drive_link = response_data.get('Google_Drive_Link', '')
            if google_drive_link:
                print(f"    Google Drive Link: {google_drive_link}")
            else:
                print(f"     Google Drive Link: Not available")
            
            # Full response for debugging
            print(f"\n Full Response:")
            print(json.dumps(response_data, indent=2))
            
        else:
            print(f" Test failed with status: {response.status_code}")
            print(f"Response: {response.text}")
            
    except Exception as e:
        print(f" Test failed with exception: {str(e)}")
        
else:
    print(" No files available for testing")